# Norwegian Newspaper Scraper (nb.no)
This notebook automates the discovery and downloading of high-resolution newspaper archives from `nb.no`.

**Workflow:**
1. **Setup:** Configure Chrome to download files silently to a specific folder.
2. **Helpers:** Load utility functions for parsing dates, waiting for downloads, and renaming files.
3. **Phase 1 (Discovery):** Scroll through the search results and collect all unique newspaper URLs.
4. **Phase 2 (Extraction):** Visit each URL directly, trigger the high-quality download, wait for completion, and rename the file.

In [1]:
import os
import time
import re
import glob
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- CONFIGURATION ---
# MUST be an absolute path! Chrome gets confused by relative paths
download_dir = os.path.abspath("scans/raw") 
SEARCH_URL = "https://www.nb.no/search?mediatype=aviser&sort=dateasc&series.exact=Sogns%20Avis&fromDate=19460101&toDate=19571231"

# Ensure download directory exists
os.makedirs(download_dir, exist_ok=True)

chrome_options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False, # Silences the "Save As" popup
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
    "plugins.always_open_pdf_externally": True # Forces PDF download instead of opening in viewer
}
chrome_options.add_experimental_option("prefs", prefs)

# Initialize driver and wait
driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 15)

print(f"Browser launched! Downloads will be saved to: {download_dir}")

Browser launched! Downloads will be saved to: /mnt/data/development/GLM-OCR-Norwegian-Nynorsk/scans/raw


### Helper Functions
These functions handle the file system operations: waiting for the `.crdownload` temp file to disappear, formatting the Norwegian date string into standard `YYYY-MM-DD`, and fetching the newest file from the directory to rename it.

In [2]:
def wait_for_download_to_finish(download_directory, timeout=300):
    """
    Monitors the download directory. Returns True when finished, False if it times out.
    """
    print("  -> Waiting for download to complete...")
    time.sleep(2) # Give Chrome a moment to initiate transfer and create .crdownload file
    
    for _ in range(timeout):
        files = os.listdir(download_directory)
        is_downloading = any(f.endswith('.crdownload') or f.endswith('.tmp') for f in files)
        
        if not is_downloading:
            print("  -> Download finished!")
            return True
            
        time.sleep(1) # Check again in 1 second
        
    print("  -> ERROR: Download timed out!")
    return False

def format_filename(raw_text):
    """
    Converts "Sogns Avis, lørdag 5. januar 1946" into "sogns-avis-1946-01-05.pdf"
    """
    try:
        parts = raw_text.split(',')
        paper_name = parts[0].strip().lower().replace(' ', '-')
        date_str = parts[1].strip() 

        match = re.search(r"(\d+)\.?\s+([a-zA-ZæøåÆØÅ]+)\s+(\d{4})", date_str)
        
        if match:
            day = match.group(1).zfill(2) # Adds leading zero (5 -> 05)
            month_name = match.group(2).lower()
            year = match.group(3)

            months = {
                "januar": "01", "februar": "02", "mars": "03", "april": "04",
                "mai": "05", "juni": "06", "juli": "07", "august": "08",
                "september": "09", "oktober": "10", "november": "11", "desember": "12"
            }
            month = months.get(month_name, "00")
            
            return f"{paper_name}-{year}-{month}-{day}.pdf"
            
    except Exception as e:
        print(f"  -> Error formatting filename text: {e}")
        
    return "unknown-document.pdf"    

def get_latest_downloaded_file(download_dir):
    """Returns the most recently created file in the download directory."""
    files = glob.glob(os.path.join(download_dir, '*'))
    valid_files = [f for f in files if os.path.isfile(f) and not f.endswith('.crdownload') and not f.endswith('.tmp')]
    
    if not valid_files:
        return None
        
    return max(valid_files, key=os.path.getctime)

### Phase 1: Collect URLs
We navigate to the search results and scroll through the Angular virtual DOM. Instead of clicking the cards, we extract the underlying `href` link for the newspaper item. Once the page stops yielding new links, the loop finishes.

In [3]:
# Go to the search page
driver.get(SEARCH_URL)

# Wait for the initial grid to load
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "cdk-virtual-scroll-viewport")))
print("Search page loaded. Beginning scroll extraction...")

unique_urls = set()
scroll_retries = 0
max_retries = 3

while scroll_retries < max_retries:
    # Find all anchor tags within the item cards that contain '/items/' in the URL
    links = driver.find_elements(By.XPATH, "//nb-item-card//a[contains(@href, '/items/')]")
    new_urls_found = False
    
    for link in links:
        href = link.get_attribute("href")
        # Ensure it's a valid link and we haven't seen it yet
        if href and href not in unique_urls:
            unique_urls.add(href)
            new_urls_found = True
            
    if new_urls_found:
        scroll_retries = 0 # We found new content, reset retry counter
    else:
        scroll_retries += 1
        print(f"No new items seen. Scroll retry {scroll_retries}/{max_retries}")
        
    # Scroll the container to fetch more items
    scroll_container = driver.find_element(By.CSS_SELECTOR, "cdk-virtual-scroll-viewport")
    driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight;", scroll_container)
    
    # Wait for the network to render the new items
    time.sleep(2)

# Convert the set to a sorted list just to keep things orderly
newspaper_urls = sorted(list(unique_urls))
print(f"\n--- URL Collection Complete ---")
print(f"Total unique newspapers found: {len(newspaper_urls)}")

Search page loaded. Beginning scroll extraction...
No new items seen. Scroll retry 1/3
No new items seen. Scroll retry 1/3
No new items seen. Scroll retry 2/3
No new items seen. Scroll retry 3/3

--- URL Collection Complete ---
Total unique newspapers found: 484


### Phase 2: Download & Rename
Now we iterate through our collected URLs. Because we are navigating directly to the exact page, we don't have to worry about the virtual scroller breaking or losing our place in the grid.

In [4]:
print(f"Starting download sequence for {len(newspaper_urls)} items...")

for index, url in enumerate(newspaper_urls):
    try:
        print(f"\n[{index + 1}/{len(newspaper_urls)}] Navigating to: {url}")
        driver.get(url)
        
        # 1. Grab the label text for the filename
        label_element = wait.until(EC.presence_of_element_located((
            By.CSS_SELECTOR, "[data-testid='ngx-mime-manifest-label']"
        )))
        raw_label_text = label_element.text
        desired_filename = format_filename(raw_label_text)
        print(f"  -> Target filename: {desired_filename}")
        
        # 2. Click Download Menu
        download_icon = wait.until(EC.element_to_be_clickable((By.XPATH, "//mat-icon[text()='file_download']")))
        driver.execute_script("arguments[0].click();", download_icon)
        
        # 3. Select High Quality
        high_quality_span = wait.until(EC.presence_of_element_located((
            By.XPATH, "//span[contains(text(), 'Høy (for trykking og reproduksjon)')]"
        )))
        driver.execute_script("arguments[0].click();", high_quality_span)
        
        # 4. Trigger Download
        download_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-testid='download-button']")))
        driver.execute_script("arguments[0].click();", download_btn)
        print("  -> Download triggered!")
        
        # 5. Smart Wait for filesystem
        if wait_for_download_to_finish(download_dir, timeout=600):
            # 6. Rename the file
            latest_file = get_latest_downloaded_file(download_dir)
            if latest_file:
                new_file_path = os.path.join(download_dir, desired_filename)
                
                # Overwrite if file already exists
                if os.path.exists(new_file_path):
                     print(f"  -> Warning: {desired_filename} already exists. Overwriting.")
                     os.remove(new_file_path)
                     
                os.rename(latest_file, new_file_path)
                print(f"  -> Successfully saved as: {desired_filename}")
            else:
                print("  -> Warning: Could not locate the downloaded file to rename it.")
                
    except Exception as e:
        print(f"  -> Error processing item {index + 1}: {e}")
        continue # Skip to the next URL if this one fails

print("\n🎉 All downloads completed!")

Starting download sequence for 484 items...

[1/484] Navigating to: https://www.nb.no/items/00057da19b42ace08170d8bf1d2391e3?page=0
  -> Target filename: sogns-avis-1953-09-03.pdf
  -> Download triggered!
  -> Waiting for download to complete...
  -> Download finished!
  -> Successfully saved as: sogns-avis-1953-09-03.pdf

[2/484] Navigating to: https://www.nb.no/items/00a85532c3ee5a239b5e04faecf1379b?page=0
  -> Target filename: sogns-avis-1950-03-23.pdf
  -> Download triggered!
  -> Waiting for download to complete...
  -> Download finished!
  -> Successfully saved as: sogns-avis-1950-03-23.pdf

[3/484] Navigating to: https://www.nb.no/items/00fd8988297132c0b3928257e5d0e02d?page=0
  -> Target filename: sogns-avis-1954-05-20.pdf
  -> Download triggered!
  -> Waiting for download to complete...
  -> Download finished!
  -> Successfully saved as: sogns-avis-1954-05-20.pdf

[4/484] Navigating to: https://www.nb.no/items/01b1ca0cb94111a86675248b2a727d0b?page=0
  -> Target filename: sogns-

KeyboardInterrupt: 